# 05 — Proposed system: Deterministic–LLM hybrid router (Second Review)

Evaluates the proposed router on MCPBench **without new LLM calls**: Tier 1 rules + consensus gate are computed here, and escalated queries reuse the per-query Hi-RAG / Full / ColBERT results already saved on Drive by notebooks 02–03.

τ is calibrated on the LLM-generated Stage 2 queries (`train_queries.json`), never on the test queries.

Runtime: Tier 1 + BM25 gate on CPU in seconds; the Stage 1 (bge-large-en) gate needs the GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RESULTS = '/content/drive/MyDrive/hirag_results_or'          # e2e_*.json from notebooks 02/03
CALIB   = '/content/drive/MyDrive/hirag_stage2/train_queries.json'
OUT     = '/content/drive/MyDrive/hirag_router'
import os, glob; os.makedirs(OUT, exist_ok=True)
print(sorted(os.path.basename(f) for f in glob.glob(RESULTS + '/e2e_*.json')))

In [ ]:
# upload hirag-repro.zip (this version) to Drive or the session, then:
!unzip -q -o /content/drive/MyDrive/hirag-repro.zip -d /content && cd /content/hirag-repro && pip -q install -r requirements.txt
%cd /content/hirag-repro
!python -m pytest tests/test_router.py -q

## 1. Main result — consensus gate with BM25 (CPU), all fallbacks

In [ ]:
!python scripts/run_router.py --retriever bm25 --fallback-dir $RESULTS --calib-file $CALIB --eps 0.05 --out-dir $OUT
!python scripts/run_router.py --retriever bm25 --fallback-dir $RESULTS --calib-file $CALIB --eps 0.10 --out-dir $OUT

## 2. Consensus gate with Hi-RAG Stage 1 (BM25 + bge-large-en) — GPU

In [ ]:
!python scripts/run_router.py --retriever stage1 --embed-model BAAI/bge-large-en --fallback-dir $RESULTS --calib-file $CALIB --eps 0.05 --out-dir $OUT
!python scripts/run_router.py --retriever stage1 --embed-model BAAI/bge-large-en --fallback-dir $RESULTS --calib-file $CALIB --eps 0.10 --out-dir $OUT

## 3. Ablations — rules-only gate, no orchestrator

In [ ]:
!python scripts/run_router.py --retriever none  --fallback-dir $RESULTS --calib-file $CALIB --eps 0.10 --out-dir $OUT
!python scripts/run_router.py --retriever bm25  --no-split --fallback-dir $RESULTS --calib-file $CALIB --eps 0.10 --out-dir $OUT

## 4. Accuracy vs LLM-call rate (figure for the slides / paper)

In [ ]:
import json, matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, kind in zip(axes, ['single', 'multi']):
    for tag, label in [('router_bm25_gencalib', 'Router (rules + BM25 gate)'), ('router_stage1_gencalib', 'Router (rules + Stage 1 gate)')]:
        p = f'{OUT}/{tag}.json'
        if not os.path.exists(p): continue
        cv = json.load(open(p))['curve']
        xs = [r.get(f'{kind}_hirag_llm_rate') for r in cv]; ys = [r.get(f'{kind}_hirag_router_acc') for r in cv]
        pts = sorted((x, y) for x, y in zip(xs, ys) if x is not None)
        ax.plot([x for x, _ in pts], [y for _, y in pts], 'o-', label=label)
        base = cv[0].get(f'{kind}_hirag_base_acc')
        full = cv[0].get(f'{kind}_full_base_acc')
    if base: ax.scatter([100], [base], marker='s', s=60, label='Hi-RAG (100% LLM calls)', zorder=3)
    if full: ax.scatter([100], [full], marker='^', s=60, label='Full context (100% LLM calls)', zorder=3)
    ax.set_xlabel('LLM-call rate (%)'); ax.set_ylabel('Accuracy (%)'); ax.set_title(f'MCPBench {kind}-service')
    ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/router_tradeoff.png', dpi=200); plt.show()

## 5. Audit trail — why a query was (or was not) routed without the LLM

In [ ]:
d = json.load(open(f'{OUT}/router_bm25_gencalib.json'))
for x in [x for x in d['decisions'] if x['tier'] == 'deterministic'][:5] + [x for x in d['decisions'] if x['tier'] == 'fallback'][:3]:
    print(x['tier'].upper(), round(x['confidence'], 3), '|', x['query'][:90])
    for s in x['subs']: print('    ->', s['tool'], '| retriever:', s['retriever_tool'], '| evidence:', {k: round(v, 2) for k, v in s['evidence'].items()})

## 6. Second benchmark: ToolBench (StableToolBench G1/G2/G3, 10k+ tools) — scalability and gate safety
No LLM, binary NDCG@1/3/5 as in the reproduction's ToolLLM test. τ from cross-fitting on ToolBench and transferred from MCPBench.

In [ ]:
!python scripts/run_router_toolbench.py --retriever bm25 --out-dir $OUT
!python scripts/run_router_toolbench.py --retriever stage1 --embed-model BAAI/bge-large-en --out-dir $OUT

## 7. Self-distillation with the main settings (CPU, minutes)

In [ ]:
for m in ['hirag', 'full']:
    !python scripts/run_distill.py --calib-file $CALIB --fallback-dir $RESULTS --fallback-method {m} --compose-multi 400 --eps 0.10,0.15 --out-dir $OUT
    !python scripts/run_distill.py --calib-file $CALIB --fallback-dir $RESULTS --fallback-method {m} --compose-multi 400 --eps 0.10,0.15 --noise 0.15 --out-dir $OUT

## 8. TOMORROW (GPU + OpenRouter key): label the log with the LLM's REAL decisions (~1,200 calls, cached)

In [ ]:
import os; from google.colab import userdata; os.environ['LLM_API_KEY'] = userdata.get('OPENROUTER_API_KEY')   # key from Colab Secrets
!python scripts/label_logs.py --train-file $CALIB --stage2 /content/drive/MyDrive/hirag_stage2/stage2.pt \
    --model qwen/qwen3-32b --extra-body '{"reasoning": {"enabled": false}}' \
    --cache /content/drive/MyDrive/hirag_llm_cache.jsonl --out /content/drive/MyDrive/hirag_stage2/train_queries_llm_labels.json
!python scripts/run_distill.py --calib-file /content/drive/MyDrive/hirag_stage2/train_queries_llm_labels.json \
    --fallback-dir $RESULTS --compose-multi 400 --eps 0.10,0.15 --out-dir $OUT/llm_labels

## 9. Second LLM (robustness): Hi-RAG and Full context with another model, then the router on top
Pick any OpenRouter model with tool calling (e.g. a Llama-3.3-70B instruct). ~520 calls for hirag; Full context is long (17k tokens/query) — check the price first.

In [ ]:
M2 = 'meta-llama/llama-3.3-70b-instruct'; R2 = '/content/drive/MyDrive/hirag_results_llm2'
C2 = f'--model {M2} --extra-body {{}} --cache /content/drive/MyDrive/hirag_llm_cache.jsonl --embed-model BAAI/bge-large-en --max-tokens 128 --min-interval 1 --out-dir {R2}'
S2 = '--stage2 /content/drive/MyDrive/hirag_stage2/stage2.pt'
!python scripts/run_e2e.py --method hirag --split single {S2} {C2}
!python scripts/run_e2e.py --method hirag --split multi  {S2} {C2}
!python scripts/run_distill.py --calib-file $CALIB --fallback-dir {R2} --fallback-model llama-3.3-70b --compose-multi 400 --eps 0.10,0.15 --out-dir $OUT/llm2